# 03 · Predictive Modeling
**Goal:** predict *before the appointment* which patients are likely to miss, so staff can act (extra reminder, phone call, overbooking, easy rescheduling).

Plan
1. Prepare features (only information known at booking time - no leakage)
2. Stratified train / test split
3. Compare models: Dummy, Logistic Regression, Random Forest, XGBoost
4. Tune the best model
5. Evaluate honestly (ROC-AUC, PR-AUC, recall, precision, F1, calibration)
6. Business view: *if we only call the riskiest 20% of patients, how many no-shows do we reach?*
7. Explain the model with SHAP
8. Save model, metrics and predictions for the dashboard

**Why not accuracy?** 80% of patients attend. A model that predicts "everyone attends" scores 80% accuracy and is useless.

In [1]:
import sys, json, warnings, joblib
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import data_prep as dp

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score, precision_score, f1_score,
                             roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, brier_score_loss)
from xgboost import XGBClassifier
import shap

SEED = 42
BLUE, RED, GREY = "#2E86AB", "#E4572E", "#9AA5B1"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
FIG = "../reports/figures/"
def save(name):
    plt.tight_layout(); plt.savefig(FIG + name + ".png", dpi=160, bbox_inches="tight"); plt.show()

df = pd.read_csv("../data/processed/appointments_clean.csv", parse_dates=["scheduled_dt", "appointment_date", "scheduled_date"])
print(df.shape, f"| no-show rate {df['missed'].mean():.1%}")

(110521, 32) | no-show rate 20.2%


## 1 · Features and split
All features are known when the appointment is booked:
`age, wait_days, scheduled_hour, appointment_dow, prior_appts, prior_noshows, num_conditions`, the yes/no flags, and `neighbourhood`.

`neighbourhood` has 81 values. Instead of 81 dummy columns we use **target encoding** (each neighbourhood is replaced by its smoothed historical no-show rate). Scikit-learn's `TargetEncoder` fits this with internal cross-fitting **inside the pipeline**, so the test set is never used to build it.

In [2]:
NUM = dp.NUMERIC_FEATURES + dp.BINARY_FEATURES
CAT = dp.CATEGORICAL_FEATURES
FEATURES = NUM + CAT
X, y = df[FEATURES], df["missed"]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.20, stratify=y, random_state=SEED)
print(f"Train {len(X_train):,} | Test {len(X_test):,}")
print(f"No-show rate  train {y_train.mean():.3f} | test {y_test.mean():.3f}   (stratified -> same)")

# Patients can appear several times. Check how much the train and test sets share patients.
shared = df.loc[idx_test, "patient_id"].isin(df.loc[idx_train, "patient_id"]).mean()
print(f"Share of test appointments whose patient also appears in train: {shared:.1%}")

Train 88,416 | Test 22,105
No-show rate  train 0.202 | test 0.202   (stratified -> same)
Share of test appointments whose patient also appears in train: 60.0%


The same patient can appear in both train and test. Most patients have only one appointment, but a repeat patient's behaviour in training could make the test slightly easier. Section 5 re-checks the result with a **patient-grouped split** to make sure the score is not inflated.

In [3]:
def preprocessor(scale=False):
    num_step = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("nbh", TargetEncoder(target_type="binary", smooth="auto", random_state=SEED), CAT),
        ("num", num_step, NUM)], verbose_feature_names_out=False).set_output(transform="pandas")

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()   # ~4 : handles class imbalance
models = {
    "Dummy (always attends)": Pipeline([("pre", preprocessor()), ("m", DummyClassifier(strategy="prior"))]),
    "Logistic Regression": Pipeline([("pre", preprocessor(True)), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Random Forest": Pipeline([("pre", preprocessor()), ("m", RandomForestClassifier(
        n_estimators=200, min_samples_leaf=30, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
    "XGBoost": Pipeline([("pre", preprocessor()), ("m", XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight, eval_metric="logloss", n_jobs=1, random_state=SEED))]),
}
print("class weight (attended per missed):", round(pos_weight, 2))

class weight (attended per missed): 3.95


## 2 · Cross-validated comparison (training data only)

In [4]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_rows = []
for name, pipe in models.items():
    auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    ap = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=1)
    cv_rows.append({"model": name, "ROC-AUC (cv mean)": auc.mean(), "ROC-AUC (cv std)": auc.std(), "PR-AUC (cv mean)": ap.mean()})
cv_table = pd.DataFrame(cv_rows).set_index("model").round(4)
cv_table

,ROC-AUC (cv mean),ROC-AUC (cv std),PR-AUC (cv mean)
model,,,
Dummy (always attends),0.5000,0.0000,0.2019
Logistic Regression,0.7248,0.0029,0.3472
Random Forest,0.7420,0.0038,0.3791
XGBoost,0.7427,0.0037,0.3808


## 3 · Hyper-parameter tuning for XGBoost

In [5]:
param_dist = {
    "m__n_estimators": [200, 300, 500],
    "m__max_depth": [3, 4, 5, 6],
    "m__learning_rate": [0.02, 0.05, 0.1],
    "m__min_child_weight": [1, 5, 20, 50],
    "m__subsample": [0.7, 0.8, 1.0],
    "m__colsample_bytree": [0.6, 0.8, 1.0],
    "m__reg_lambda": [1, 5, 20],
}
search = RandomizedSearchCV(models["XGBoost"], param_dist, n_iter=10, scoring="roc_auc", cv=3,
                            random_state=SEED, n_jobs=1, refit=True)
search.fit(X_train, y_train)
print("Best CV ROC-AUC:", round(search.best_score_, 4))
print({k.replace("m__", ""): v for k, v in search.best_params_.items()})
models["XGBoost (tuned)"] = search.best_estimator_

Best CV ROC-AUC: 0.743
{'subsample': 0.7, 'reg_lambda': 1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.02, 'colsample_bytree': 0.8}


## 4 · Evaluation on the untouched test set

In [6]:
def evaluate(pipe, thr=0.5):
    p = pipe.predict_proba(X_test)[:, 1]; pred = (p >= thr).astype(int)
    return p, {"ROC-AUC": roc_auc_score(y_test, p), "PR-AUC": average_precision_score(y_test, p),
               "Recall": recall_score(y_test, pred, zero_division=0), "Precision": precision_score(y_test, pred, zero_division=0),
               "F1": f1_score(y_test, pred, zero_division=0), "Accuracy": (pred == y_test).mean()}

probs, rows = {}, {}
for name, pipe in models.items():
    if name != "XGBoost (tuned)":      # the tuned model was already refit by the search
        pipe.fit(X_train, y_train)
    probs[name], rows[name] = evaluate(pipe)
results = pd.DataFrame(rows).T.round(3)
print("Threshold 0.50  (class-weighted models, so 0.5 already favours recall)")
results

Threshold 0.50  (class-weighted models, so 0.5 already favours recall)


,ROC-AUC,PR-AUC,Recall,Precision,F1,Accuracy
Dummy (always attends),0.500,0.202,0.000,0.000,0.000,0.798
Logistic Regression,0.721,0.345,0.865,0.299,0.444,0.563
Random Forest,0.735,0.373,0.791,0.317,0.452,0.613
XGBoost,0.735,0.372,0.821,0.310,0.450,0.594
XGBoost (tuned),0.736,0.380,0.804,0.313,0.451,0.604


Reading the table
- The **dummy model** has about 80% accuracy yet **0 recall** - it catches no no-shows. This is why accuracy is misleading here.
- The real models trade accuracy for **recall** (finding the people who will miss). Compare them on **ROC-AUC** and **PR-AUC**, which do not depend on one threshold.
- A ROC-AUC around 0.7 is realistic for this dataset: behaviour is only partly explained by the recorded variables.

In [7]:
best_name = results.drop("Dummy (always attends)")["ROC-AUC"].idxmax()
print("Best model on test ROC-AUC:", best_name)
best = models[best_name]; p_best = probs[best_name]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
pal = {"Logistic Regression": GREY, "Random Forest": "#7FB069", "XGBoost": "#E6AA68", "XGBoost (tuned)": BLUE}
for n, p in probs.items():
    if n.startswith("Dummy"): continue
    fpr, tpr, _ = roc_curve(y_test, p); ax[0].plot(fpr, tpr, color=pal[n], lw=2.5 if n == best_name else 1.5, label=f"{n} (AUC {roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p); ax[1].plot(rc, pr, color=pal[n], lw=2.5 if n == best_name else 1.5, label=f"{n} (AP {average_precision_score(y_test, p):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate (recall)"); ax[0].set_title("ROC curves"); ax[0].legend(frameon=False, fontsize=8)
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label="Random guess"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall curves"); ax[1].legend(frameon=False, fontsize=8)
save("12_roc_pr_curves")

Best model on test ROC-AUC: XGBoost (tuned)


## 5 · Robustness checks
1. **Patient-grouped cross-validation** - no patient is in both training and validation folds.
2. **Time-based split** - train on earlier appointment dates, test on the latest ones (closest to real deployment).

If the scores stay close to the random split, we can trust the result.

In [8]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
g = GroupKFold(n_splits=5)
scores = cross_val_score(clone(best), X, y, cv=g, groups=df["patient_id"], scoring="roc_auc")
print(f"Patient-grouped 5-fold ROC-AUC: {scores.mean():.3f} +/- {scores.std():.3f}")

cut = df["appointment_date"].quantile(0.8)
tr, te = df["appointment_date"] <= cut, df["appointment_date"] > cut
m_time = clone(best).fit(X[tr], y[tr]); auc_time = roc_auc_score(y[te], m_time.predict_proba(X[te])[:, 1])
print(f"Time-based split (train up to {cut.date()}, test after): ROC-AUC {auc_time:.3f}  | train n={tr.sum():,}, test n={te.sum():,}")
print(f"Random split ROC-AUC (for comparison)                  : {roc_auc_score(y_test, p_best):.3f}")

Patient-grouped 5-fold ROC-AUC: 0.740 +/- 0.003


Time-based split (train up to 2016-06-01, test after): ROC-AUC 0.731  | train n=88,535, test n=21,986
Random split ROC-AUC (for comparison)                  : 0.736


## 6 · Choosing a threshold - a business decision, not a statistical one

In [9]:
prec, rec, thr = precision_recall_curve(y_test, p_best)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
t_f1 = thr[np.argmax(f1s[:-1])]
t_rec = thr[np.argmax(rec[:-1] < 0.60) - 1]    # highest threshold that still keeps recall >= 60%

def at(t):
    pred = (p_best >= t).astype(int)
    return {"threshold": round(float(t), 3), "flagged_%": pred.mean() * 100, "recall": recall_score(y_test, pred),
            "precision": precision_score(y_test, pred), "f1": f1_score(y_test, pred)}
thr_table = pd.DataFrame({"Default 0.50": at(0.5), "Best F1": at(t_f1), "Recall >= 60%": at(t_rec)}).T.round(3)
thr_table

,threshold,flagged_%,recall,precision,f1
Default 0.50,0.500,51.875,0.804,0.313,0.451
Best F1,0.515,49.265,0.778,0.319,0.452
Recall >= 60%,0.588,35.146,0.600,0.345,0.438


In [10]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(thr, prec[:-1], color=BLUE, label="Precision"); ax.plot(thr, rec[:-1], color=RED, label="Recall"); ax.plot(thr, f1s[:-1], color=GREY, label="F1")
ax.axvline(t_f1, color="k", ls=":", lw=1); ax.set_xlabel("Risk threshold"); ax.set_ylabel("Score"); ax.set_title("Threshold trade-off: catch more no-shows vs. more false alarms"); ax.legend(frameon=False)
save("13_threshold_tradeoff")

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test, (p_best >= t_f1).astype(int)), display_labels=["Attended", "Missed"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format=",")
ax.set_title(f"Confusion matrix (threshold {t_f1:.2f})"); ax.grid(False); save("14_confusion_matrix")

### Business view: gains and lift
Staff can rarely contact every patient. If they phone only the riskiest 10 / 20 / 30% of upcoming appointments, how many of the real no-shows do they reach?

In [11]:
d = pd.DataFrame({"p": p_best, "y": y_test.values}).sort_values("p", ascending=False).reset_index(drop=True)
d["decile"] = pd.qcut(d.index, 10, labels=range(1, 11))
dec = d.groupby("decile", observed=True).agg(patients=("y", "size"), no_shows=("y", "sum"), actual_rate=("y", "mean"), avg_raw_score=("p", "mean"))
dec["cum_capture_%"] = dec["no_shows"].cumsum() / dec["no_shows"].sum() * 100
dec["lift"] = dec["actual_rate"] / y_test.mean()
print("avg_raw_score = uncalibrated, class-weighted model output. It is only meant for RANKING (it overstates real probabilities); calibrated probabilities follow below.")
dec.round(3)

avg_raw_score = uncalibrated, class-weighted model output. It is only meant for RANKING (it overstates real probabilities); calibrated probabilities follow below.


,patients,no_shows,actual_rate,avg_raw_score,cum_capture_%,lift
decile,,,,,,
1,2211,962,0.435,0.748,21.555,2.155
2,2210,754,0.341,0.681,38.449,1.690
3,2211,646,0.292,0.635,52.924,1.447
4,2210,591,0.267,0.588,66.166,1.325
5,2211,549,0.248,0.538,78.467,1.230
6,2210,439,0.199,0.480,88.304,0.984
7,2210,297,0.134,0.333,94.959,0.666
8,2211,89,0.040,0.152,96.953,0.199
9,2210,83,0.038,0.111,98.812,0.186


In [12]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(dec.index.astype(str), dec["actual_rate"] * 100, color=BLUE); ax[0].axhline(y_test.mean() * 100, color=RED, ls="--", label="Average")
ax[0].set_xlabel("Risk decile (1 = riskiest 10%)"); ax[0].set_ylabel("Actual no-show rate (%)"); ax[0].set_title("The model separates low- from high-risk patients"); ax[0].legend(frameon=False)
x = np.r_[0, np.arange(1, 11) * 10]; yv = np.r_[0, dec["cum_capture_%"].values]
ax[1].plot(x, yv, color=BLUE, marker="o", label="Model"); ax[1].plot([0, 100], [0, 100], "k--", label="Random"); ax[1].set_xlabel("% of appointments contacted (riskiest first)"); ax[1].set_ylabel("% of all no-shows reached")
ax[1].set_title("Cumulative gains"); ax[1].legend(frameon=False)
save("15_gains_lift")
top20 = dec.loc[[1, 2], "no_shows"].sum() / dec["no_shows"].sum()
print(f"Contacting the riskiest 20% of appointments reaches {top20:.0%} of all no-shows (random would reach 20%).  Top-decile lift: {dec.loc[1, 'lift']:.1f}x")

Contacting the riskiest 20% of appointments reaches 38% of all no-shows (random would reach 20%).  Top-decile lift: 2.2x


### Calibration - are predicted probabilities trustworthy?
The model was trained with class weights, which inflates probabilities. We fit a simple **calibration** step on the training data (cross-validated isotonic regression) so a risk of '30%' really means about 30%.

In [13]:
from sklearn.calibration import CalibratedClassifierCV
cal = CalibratedClassifierCV(clone(best), method="isotonic", cv=5).fit(X_train, y_train)
p_cal = cal.predict_proba(X_test)[:, 1]
print(f"Brier score  raw {brier_score_loss(y_test, p_best):.4f}  ->  calibrated {brier_score_loss(y_test, p_cal):.4f}  (lower is better)")
print(f"ROC-AUC      raw {roc_auc_score(y_test, p_best):.4f}  ->  calibrated {roc_auc_score(y_test, p_cal):.4f}  (ranking is essentially unchanged)")

fig, ax = plt.subplots(figsize=(5.5, 5))
for lab, p, col in [("Raw (class-weighted)", p_best, GREY), ("Calibrated", p_cal, BLUE)]:
    fr, mp = calibration_curve(y_test, p, n_bins=10, strategy="quantile"); ax.plot(mp, fr, marker="o", color=col, label=lab)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect"); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed no-show rate"); ax.set_title("Calibration curve"); ax.legend(frameon=False)
save("16_calibration")

Brier score  raw 0.2087  ->  calibrated 0.1428  (lower is better)
ROC-AUC      raw 0.7364  ->  calibrated 0.7367  (ranking is essentially unchanged)


## 7 · Explaining the model (feature importance and SHAP)

In [14]:
xgb_model = best.named_steps["m"]; pre = best.named_steps["pre"]
Xt = pre.transform(X_test)
sample = Xt.sample(4000, random_state=SEED)
explainer = shap.TreeExplainer(xgb_model); sv = explainer(sample)

nice = {"neighbourhood": "neighbourhood risk"}
sv.feature_names = [nice.get(f, f) for f in sample.columns]
plt.figure(); shap.plots.beeswarm(sv, max_display=15, show=False); plt.title("SHAP: what drives predicted no-show risk"); save("17_shap_beeswarm")

In [15]:
imp = pd.Series(np.abs(sv.values).mean(0), index=sv.feature_names).sort_values()
fig, ax = plt.subplots(figsize=(8, 5.5)); ax.barh(imp.index, imp.values, color=BLUE)
ax.set_xlabel("Mean |SHAP value| (average impact on risk, log-odds)"); ax.set_title("Global feature importance"); save("18_feature_importance")
imp.sort_values(ascending=False).round(3).head(10).to_frame("mean |SHAP|")

,mean |SHAP|
wait_days,0.754
age,0.247
same_day,0.191
neighbourhood risk,0.090
sms_received,0.076
scheduled_hour,0.071
prior_noshows,0.066
prior_appts,0.060
appointment_dow,0.036
scholarship,0.029


In [16]:
# Dependence plots for the top two drivers
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
shap.plots.scatter(sv[:, "wait_days"], ax=ax[0], show=False, color=sv[:, "age"]); ax[0].set_title("Waiting days -> risk")
shap.plots.scatter(sv[:, "age"], ax=ax[1], show=False, color=sv[:, "wait_days"]); ax[1].set_title("Age -> risk")
save("19_shap_dependence")

## 8 · Save everything for the dashboard

In [17]:
# 1. Calibrated model (used by the dashboard's single-patient risk calculator)
final_model = CalibratedClassifierCV(clone(best), method="isotonic", cv=5).fit(X, y)   # refit on ALL data for deployment
joblib.dump({"model": final_model, "features": FEATURES, "num": NUM, "cat": CAT, "threshold_f1": float(t_f1)}, "../models/noshow_model.joblib")

# 2. Test-set predictions with calibrated probabilities (honest out-of-sample scores)
test_pred = df.loc[idx_test].copy()
test_pred["risk_score"] = p_cal
q = np.quantile(test_pred["risk_score"], [0.6, 0.85])
test_pred["risk_tier"] = pd.cut(test_pred["risk_score"], [-1, q[0], q[1], 2], labels=["Low", "Medium", "High"])
test_pred["risk_decile"] = pd.qcut(test_pred["risk_score"].rank(method="first", ascending=False), 10, labels=range(1, 11)).astype(int)
test_pred.to_csv("../data/processed/test_predictions.csv", index=False)

# 3. Metrics
cal_row = {"ROC-AUC": roc_auc_score(y_test, p_cal), "PR-AUC": average_precision_score(y_test, p_cal)}
metrics = {"best_model": best_name, "test_results": results.to_dict(orient="index"),
           "cv_results": cv_table.to_dict(orient="index"), "calibrated": cal_row,
           "grouped_cv_auc": float(scores.mean()), "time_split_auc": float(auc_time),
           "threshold_table": thr_table.to_dict(orient="index"), "top20_capture": float(top20),
           "top_decile_lift": float(dec.loc[1, "lift"]), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
           "base_rate": float(y.mean()), "best_params": {k.replace("m__", ""): (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()},
           "top_features": imp.sort_values(ascending=False).head(8).round(4).to_dict()}
json.dump(metrics, open("../reports/model_metrics.json", "w"), indent=2, default=float)
dec.round(4).to_csv("../reports/decile_table.csv")
results.to_csv("../reports/model_comparison.csv")
print("Saved: models/noshow_model.joblib, data/processed/test_predictions.csv, reports/model_metrics.json")
test_pred[["appointment_id", "age", "wait_days", "neighbourhood", "risk_score", "risk_tier", "missed"]].head()

Saved: models/noshow_model.joblib, data/processed/test_predictions.csv, reports/model_metrics.json


,appointment_id,age,wait_days,neighbourhood,risk_score,risk_tier,missed
82806,5583315,53,18,BONFIM,0.271186,Medium,1
58886,5725340,60,4,CONQUISTA,0.223342,Low,0
107933,5691555,8,20,RESISTÊNCIA,0.324135,Medium,0
79677,5631368,64,12,CENTRO,0.170993,Low,0
62674,5682650,34,0,JARDIM CAMBURI,0.025734,Low,0


## 9 · Conclusions and limits
- A tuned gradient-boosting model ranks patients by no-show risk clearly better than chance (see ROC-AUC above) and the riskiest 20% of appointments contain a disproportionate share of all no-shows.
- The strongest drivers are **waiting time, age, SMS (targeting), patient history and neighbourhood**.
- Scores held up under a **patient-grouped** and a **time-based** check, so they are not an artefact of the random split.
- **Limits:** only ~6 weeks of data; no information on transport, weather, appointment type, doctor or reason for visit; observational data - the model *predicts*, it does not prove *causes*. Moderate AUC means it should **prioritise outreach**, not be used to deny anyone care.
- **Fairness:** age, gender and welfare status are sensitive. Use predictions only to *add* support (reminders, easier rescheduling), never to penalise patients.